<a href="https://colab.research.google.com/github/NarutoxMessi/Capstone-Project/blob/notebooks/part4_llm_feature.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 4 – LLM Feature (Track C)
This notebook loads `best_model.pkl`, accepts feature values, predicts, and generates a natural-language explanation.

In [14]:
import sys
!{sys.executable} -m pip install scikit-learn==1.8.0

# After running this cell, you will likely need to restart the Colab runtime (Runtime -> Restart runtime) for the changes to take effect.

In [15]:
from google.colab import files

print("Please upload your 'best_model.pkl' file:")
uploaded = files.upload()

for fn in uploaded.keys():
  print(f"User uploaded file \"{fn}\" with length {len(uploaded[fn])} bytes")

Please upload your 'best_model.pkl' file:


Saving best_model.pkl to best_model (1).pkl
User uploaded file "best_model (1).pkl" with length 153730 bytes


In [23]:
import joblib
import pandas as pd

# Ensure the correct filename is used. If Colab renamed it during upload,
# you might need to adjust 'best_model.pkl' to the actual uploaded filename,
# e.g., 'best_model (1).pkl'.
# Based on the uploaded variable, the file might be named 'best_model (1).pkl'.
# Let's try to load that first, and if it fails, fallback to 'best_model.pkl'.

model_filename = 'best_model.pkl'
if 'uploaded' in locals() and 'best_model (1).pkl' in uploaded.keys():
    model_filename = 'best_model (1).pkl'

model = joblib.load(model_filename)
print(f"Model '{model_filename}' loaded successfully:")
print(model)

Model 'best_model (1).pkl' loaded successfully:
Pipeline(steps=[('simpleimputer', SimpleImputer(strategy='median')),
                ('standardscaler', StandardScaler()),
                ('randomforestclassifier',
                 RandomForestClassifier(max_depth=5, min_samples_leaf=5,
                                        random_state=42))])


In [34]:
from sklearn.metrics import accuracy_score
import numpy as np

# Assuming the target variable in your test data is named 'target' (please change if different)
target_column_name = 'target' # <<< YOU MUST CHANGE THIS TO YOUR ACTUAL TARGET COLUMN NAME

# Create missing features in test_df to match model's expected features
# 1. price_range = high - low
test_df['price_range'] = test_df['high'] - test_df['low']

# 2. daily_return = (open - previous_open) / previous_open. For simplicity, assume 0 for first row.
#    Or, if we don't have previous day's open, a simple (close-open)/open ratio might be used for daily change.
#    Given current columns, let's use a simplified daily return (close-open)/open, assuming 'open' is today's open
#    and 'close' could be represented by 'open' of the next day or simply 'high'/'low' for a daily change indication.
#    For demonstration, let's calculate (high-low)/open as a proxy if 'close' is not available.
#    A more accurate daily_return often needs sequential data which is not guaranteed for a generic test_df.
#    For now, let's assume 'daily_return' could be (high - low) / open as a simple daily volatility.
#    Alternatively, if we are predicting 'up'/'down', daily_return would be derived from sequential 'close' prices.
#    Given the context of RandomForest, and a simple numeric 'daily_return' feature expected by the model,
#    we will use (high - low) / open as a temporary placeholder.
#    A better way would be to ensure 'daily_return' is part of the uploaded test_data.csv.
#    For the generated test_df, it was directly provided.

# Let's check if we can compute daily_return. If not, we will need the user to provide it.
# For now, let's use a placeholder if not present, and assume it needs to be provided by user or original data.
if 'daily_return' not in test_df.columns:
    # Placeholder: if `daily_return` is truly missing, this might lead to inaccurate predictions.
    # Ideally, it should be in the uploaded data or derived with proper sequential data.
    test_df['daily_return'] = (test_df['high'] - test_df['low']) / test_df['open']

# 3. One-hot encode 'day_of_week'
# Ensure day_of_week is numeric (0=Monday, 1=Tuesday, ..., 6=Sunday)
day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
for i, day_name in enumerate(day_names):
    test_df[f'day_name_{day_name}'] = (test_df['day_of_week'] == i).astype(int)

# Drop original 'day' and 'day_of_week' columns if they are not in model.feature_names_in_
# and if they were used for deriving new features.
# Also drop 'day' as it's not a model feature.
columns_to_drop = []
if 'day' in test_df.columns and 'day' not in model.feature_names_in_:
    columns_to_drop.append('day')
if 'day_of_week' in test_df.columns and 'day_of_week' not in model.feature_names_in_:
    columns_to_drop.append('day_of_week')
test_df = test_df.drop(columns=columns_to_drop, errors='ignore')

# It's crucial that X_test contains the same features in the same order as model.feature_names_in_.
feature_names = model.feature_names_in_ # Get feature names directly from the loaded model

# Check if all required features are now in test_df
missing_features_for_model = [f for f in feature_names if f not in test_df.columns]
if missing_features_for_model:
    raise ValueError(f"After preprocessing, the following features are still missing from test_df: {missing_features_for_model}. Please ensure your original data or preprocessing steps create them.")


# Now, recheck the target column
if target_column_name not in test_df.columns:
    raise ValueError(f"Target column '{target_column_name}' not found in the preprocessed test data. Please update `target_column_name` variable in the code or ensure your test data contains it.")

# Separate features and target variable
X_test = test_df[feature_names]
y_test = test_df[target_column_name]

# Make predictions on the test set
y_pred = model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

print(f"Model Accuracy on Test Data: {accuracy:.4f}")

KeyError: 'day_of_week'

In [27]:
import joblib
import pandas as pd

model = joblib.load('best_model.pkl')
print(model)

Pipeline(steps=[('simpleimputer', SimpleImputer(strategy='median')),
                ('standardscaler', StandardScaler()),
                ('randomforestclassifier',
                 RandomForestClassifier(max_depth=5, min_samples_leaf=5,
                                        random_state=42))])


In [17]:
import numpy as np

# Get the feature names from the model
feature_names = model.feature_names_in_
print("Model expects the following features in this order:")
print(feature_names)

Model expects the following features in this order:
['open' 'high' 'low' 'volume' 'year' 'month' 'price_range' 'daily_return'
 'volume_category' 'day_name_Monday' 'day_name_Thursday'
 'day_name_Tuesday' 'day_name_Wednesday']


In [26]:
import numpy as np

# Get the feature names from the model
# feature_names = model.feature_names_in_ # This line is already in a previous cell

# Create the sample dictionary dynamically with correct feature names and order
sample = {feature: 0 for feature in feature_names}

# NOTE: You must replace these '0' placeholder values with your actual data.
# For example, if 'daily_return' should be 0.01:
# sample['daily_return'] = 0.01
# For example, to set 'open' to 100:
# sample['open'] = 100

X = pd.DataFrame([sample])

try:
    pred = model.predict(X)[0]
    print("Prediction:", pred)
except Exception as e:
    print("An error occurred during prediction. Please ensure all feature values are correctly set and in the correct order.")
    raise e

Prediction: 0


In [25]:
# Create the sample dictionary dynamically with correct feature names and order
sample = {feature: 0 for feature in feature_names}

# NOTE: You must replace these '0' placeholder values with your actual data.
# For example, if 'daily_return' should be 0.01:
# sample['daily_return'] = 0.01

X = pd.DataFrame([sample])

try:
    pred = model.predict(X)[0]
    print("Prediction:", pred)
except Exception as e:
    print("An error occurred during prediction. Please ensure all feature values are correctly set.")
    raise e

Prediction: 0


In [20]:
def explain_prediction(pred):
    return f"The trained Random Forest pipeline predicts class: {pred}. The prediction is produced after median imputation, feature scaling, and classification. Review the input values to understand which characteristics may have influenced the model."

print(explain_prediction(pred))

user_question = input("Ask a question about the prediction: ")
print("\nLLM-style Response:")
print(f"Question: {user_question}")
print(explain_prediction(pred))


The trained Random Forest pipeline predicts class: 0. The prediction is produced after median imputation, feature scaling, and classification. Review the input values to understand which characteristics may have influenced the model.
Ask a question about the prediction: most raised price in a day

LLM-style Response:
Question: most raised price in a day
The trained Random Forest pipeline predicts class: 0. The prediction is produced after median imputation, feature scaling, and classification. Review the input values to understand which characteristics may have influenced the model.


## Evaluate Model Accuracy

To evaluate the model, we need a test dataset. Please upload your test data as a CSV file.

In [40]:
import pandas as pd
import numpy as np

# Define the feature names and a target column based on the model's expectations
# and the previous notebook cells.
feature_names = ['open', 'high', 'low', 'volume', 'year', 'month', 'price_range', 'daily_return', 'volume_category', 'day_name_Monday', 'day_name_Thursday', 'day_name_Tuesday', 'day_name_Wednesday']
target_column_name = 'target'

# Create a dictionary with dummy data for the features and target
# You should replace these with realistic values if you want meaningful predictions.
sample_data = {
    'open': [100.0, 101.0, 99.0],
    'high': [102.0, 103.0, 100.0],
    'low': [98.0, 99.0, 97.0],
    'volume': [100000, 120000, 90000],
    'year': [2023, 2023, 2023],
    'month': [1, 1, 1],
    'price_range': [4.0, 4.0, 3.0],
    'daily_return': [0.01, 0.005, -0.002],
    'volume_category': [1, 2, 1],
    'day_name_Monday': [1, 0, 0],
    'day_name_Thursday': [0, 0, 1],
    'day_name_Tuesday': [0, 1, 0],
    'day_name_Wednesday': [0, 0, 0],
    target_column_name: [0, 1, 0] # Dummy target values
}

# Create the DataFrame
test_df = pd.DataFrame(sample_data)

# Save the DataFrame to a CSV file named 'test_data.csv'
test_df.to_csv('test_data.csv', index=False)

print("Generated 'test_data.csv' and loaded it into 'test_df'.")
display(test_df.head())

Generated 'test_data.csv' and loaded it into 'test_df'.


,open,high,low,volume,year,month,price_range,daily_return,volume_category,day_name_Monday,day_name_Thursday,day_name_Tuesday,day_name_Wednesday,target
0,100.0,102.0,98.0,100000,2023,1,4.0,0.010,1,1,0,0,0,0
1,101.0,103.0,99.0,120000,2023,1,4.0,0.005,2,0,0,1,0,1
2,99.0,100.0,97.0,90000,2023,1,3.0,-0.002,1,0,1,0,0,0


In [39]:
from google.colab import files
import pandas as pd

print("Please upload your 'test_data.csv' file:")
uploaded_test_data = files.upload()

if uploaded_test_data: # Check if any files were uploaded
    # Get the name of the first uploaded file (Colab often renames if file exists)
    uploaded_file_name = list(uploaded_test_data.keys())[0]

    if uploaded_file_name.endswith('.csv'):
        print(f"User uploaded file \"{uploaded_file_name}\" with length {len(uploaded_test_data[uploaded_file_name])} bytes")
        test_df = pd.read_csv(uploaded_file_name)
        print("Test data loaded successfully.")
        display(test_df.head())
    else:
        print(f"Error: Expected a CSV file, but received '{uploaded_file_name}'.")
        print("Please upload 'test_data.csv' again.")
else:
    print("No file was uploaded.")

Please upload your 'test_data.csv' file:


Saving test_data.csv to test_data (4).csv
User uploaded file "test_data (4).csv" with length 178 bytes
Test data loaded successfully.


,open,high,low,volume,year,month,day,day_of_week,volume_category
0,100.5,102.0,99.8,1500000,2024,7,1,0,1
1,101.2,103.1,100.7,1800000,2024,7,2,1,2
2,99.8,100.4,98.9,1200000,2024,7,3,2,0


In [7]:
print('Columns in test_df:')
print(test_df.columns)
print('\nData types of columns in test_df:')
print(test_df.info())

Columns in test_df:
Index(['open', 'high', 'low', 'volume', 'year', 'month', 'price_range',
       'daily_return', 'volume_category', 'day_name_Monday',
       'day_name_Thursday', 'day_name_Tuesday', 'day_name_Wednesday',
       'target'],
      dtype='object')

Data types of columns in test_df:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   open                3 non-null      float64
 1   high                3 non-null      float64
 2   low                 3 non-null      float64
 3   volume              3 non-null      int64  
 4   year                3 non-null      int64  
 5   month               3 non-null      int64  
 6   price_range         3 non-null      float64
 7   daily_return        3 non-null      float64
 8   volume_category     3 non-null      int64  
 9   day_name_Monday     3 non-null      int64  
 10  day_name

In [8]:
# This code inspects the columns of 'test_df'.
# IMPORTANT: This cell will only work after you have successfully uploaded 'test_data.csv'
# via cell 'a57bacc9' and 'test_df' has been created.

print('Columns in test_df:')
print(test_df.columns)
print('\nData types of columns in test_df:')
print(test_df.info())

Columns in test_df:
Index(['open', 'high', 'low', 'volume', 'year', 'month', 'price_range',
       'daily_return', 'volume_category', 'day_name_Monday',
       'day_name_Thursday', 'day_name_Tuesday', 'day_name_Wednesday',
       'target'],
      dtype='object')

Data types of columns in test_df:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   open                3 non-null      float64
 1   high                3 non-null      float64
 2   low                 3 non-null      float64
 3   volume              3 non-null      int64  
 4   year                3 non-null      int64  
 5   month               3 non-null      int64  
 6   price_range         3 non-null      float64
 7   daily_return        3 non-null      float64
 8   volume_category     3 non-null      int64  
 9   day_name_Monday     3 non-null      int64  
 10  day_name

In [29]:
print(test_df.columns.tolist())

['open', 'high', 'low', 'volume', 'year', 'month', 'day', 'day_of_week', 'volume_category']


Now that the test data is loaded, we need to separate the features (X_test) from the target variable (y_test) and then use the trained model to make predictions and evaluate its accuracy.

In [36]:
print(test_df.columns.tolist())

['open', 'high', 'low', 'volume', 'year', 'month', 'volume_category', 'target', 'daily_return', 'day_name_Monday', 'day_name_Tuesday', 'day_name_Wednesday', 'day_name_Thursday', 'day_name_Friday', 'day_name_Saturday', 'day_name_Sunday']


In [37]:
if 'price_range' in test_df.columns:
    test_df = test_df.rename(columns={'price_range': 'target'})
    print("Column 'price_range' renamed to 'target'.")
else:
    print("Column 'price_range' not found in test_df.")

print(test_df.columns.tolist())

Column 'price_range' not found in test_df.
['open', 'high', 'low', 'volume', 'year', 'month', 'volume_category', 'target', 'daily_return', 'day_name_Monday', 'day_name_Tuesday', 'day_name_Wednesday', 'day_name_Thursday', 'day_name_Friday', 'day_name_Saturday', 'day_name_Sunday']


In [41]:
from sklearn.metrics import accuracy_score

# Assuming the target variable in your test data is named 'target' (please change if different)
# You might need to adjust the feature columns to match your model's expectations.
# For example, drop any ID columns or non-feature columns.

# It's crucial that X_test contains the same features in the same order as model.feature_names_in_.
# We can enforce this by reindexing or explicitly selecting columns.

# Assuming your test_df contains both features and the target column.
# Replace 'your_target_column_name' with the actual name of your target column in test_data.csv
target_column_name = 'target' # <<< YOU MUST CHANGE THIS TO YOUR ACTUAL TARGET COLUMN NAME

if target_column_name not in test_df.columns:
    raise ValueError(f"Target column '{target_column_name}' not found in the uploaded test data. Please update `target_column_name` variable.")

# Separate features and target variable
X_test = test_df[feature_names] # Use the feature_names obtained from the model
y_test = test_df[target_column_name]

# Make predictions on the test set
y_pred = model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

print(f"Model Accuracy on Test Data: {accuracy:.4f}")

Model Accuracy on Test Data: 0.6667
